**A500 variant** — generated from the SP100 notebook by `scripts/clone_notebooks_a500.py`; do not edit directly, edit the original and re-run the script.

Requires `data/A500/raw/` built by `scripts/preprocess_a500.py` (replaces notebooks 1-2 for A500).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from datasets.SP100Stocks import SP100Stocks
from notebooks.models import TGCN, A3TGCN, DCGNN, train, measure_accuracy, get_confusion_matrix

# Stock trend classification
The goal of this task is to classify the stock movement $n$ weeks ahead as a binary up/down trend, based on historical data.

## Loading the data
The data from the custom PyG dataset for forecasting is loaded into a PyTorch dataloader.
A "transform" is applied to change the targets `y` of the dataset to a binary buy/sell class instead of the close price. 

In [ ]:
def future_close_price_to_buy_sell_class(sample: Data):
	"""
	Transforms the target y to a binary buy (1) if the stock return two weeks ahead was higher that the average market return, else sell (0)
	:param sample: Data sample
	:return: The transformed sample
	"""
	market_return = ((sample.close_price_y[:, -1] - sample.close_price[:, -1]) / sample.close_price[:, -1]).mean()
	sample.returns = ((sample.close_price_y[:, -1] - sample.close_price[:, -1]) / sample.close_price[:, -1]).unsqueeze(1)
	sample.market_return = market_return
	sample.y = (sample.returns >= 0).float()
	return sample

In [ ]:
weeks_ahead = 1

dataset = SP100Stocks(root="../data/A500/", future_window=weeks_ahead * 5, force_reload=True, transform=future_close_price_to_buy_sell_class)
dataset, dataset[0]

In [ ]:
for i in range(0, 10):
	print(f"Stock return: {dataset[i].returns[i].item() * 100:.2f}%, trend: {['Down', 'Up'][int(dataset[i].y[i].item())]}")

In [ ]:
train_part = .9
batch_size = 32

train_dataset, test_dataset = dataset[:int(train_part * len(dataset))], dataset[int(train_part * len(dataset)):]
print(f"Train dataset: {len(train_dataset)}, Test dataset: {len(test_dataset)}")
train_dataloader, test_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True), DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=True)

## Training
The previously implemented models are used, trained using the training dataset and the Adam optimizer. The `weight_decay` parameter is used for L2 regularization, to follow the T-GCN papers methodology. The loss is calculated using the Binary Cross Entropy (BCE) loss function.

In [ ]:
in_channels, out_channels, hidden_size, layers_nb, dropout = dataset[0].x.shape[-2], 1, 16, 2, .3
model = TGCN(in_channels, out_channels, hidden_size, layers_nb)

lr, weight_decay, num_epochs = 0.005, 1e-5, 100
	
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
model

In [ ]:
train(model, optimizer, criterion, train_dataloader, test_dataloader, num_epochs, "A500_UpDownTrend", measure_acc=True)

In [ ]:
torch.save(model.state_dict(), f"models/saved_models/A500_UpDownTrend_{model.__class__.__name__}.pt")

In [ ]:
model = TGCN(in_channels, out_channels, hidden_size, layers_nb)
model.load_state_dict(torch.load(f"models/saved_models/A500_UpDownTrend_{model.__class__.__name__}.pt"))

## Results

### Results on train data

In [ ]:
full_train_data = next(iter(DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=True)))
acc, cm = measure_accuracy(model, full_train_data), get_confusion_matrix(model, full_train_data)

print(f"Train accuracy: {acc * 100:.1f}%\nTrain confusion matrix:\n{cm}")

### Results on test data

In [ ]:
acc, cm = measure_accuracy(model, next(iter(train_dataloader))), get_confusion_matrix(model, next(iter(train_dataloader)))

print(f"Test accuracy: {acc * 100:.1f}%\nTest confusion matrix:\n{cm}")